# COde 01 — Features only: original or flattened Qwen images
Attach ClinDistill bank + prepared release (image_index.json) + code_full_offline_assets. GPU ON, Internet OFF, Run All. No re-splitting or raw CSV audit.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
RESEARCH_BANK_OVERRIDE = globals().get('RESEARCH_BANK_OVERRIDE')
RESEARCH_ASSETS = None

def research_find(predicate, *, include_runs=False):
    found = set()
    for base in (Path.cwd(), Path.cwd().parent / 'Data', Path('/kaggle/input')):
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if predicate(here, dirs, files):
                found.add(here.resolve())
            excluded = {'Images', 'images', '.git', '.cache', '__pycache__', '_archive', '_support',
                        'models', 'wheels', 'wheelhouse', 'official_code', '_chunks', 'student_export'}
            dirs[:] = [d for d in dirs if not d.lower().startswith('qwen')]
            if not include_runs:
                excluded.add('runs')
            dirs[:] = [d for d in dirs if d not in excluded]
            if len(here.relative_to(base).parts) >= 8:
                dirs[:] = []
    return sorted(found, key=str)

def research_one(override, found, name):
    if override is not None:
        return Path(override).resolve()
    if len(found) != 1:
        raise RuntimeError(f'{name}: found {len(found)} folders: {found}. Set the matching *_OVERRIDE.')
    return found[0]

if Path('/kaggle').exists():
    bank_root = research_one(RESEARCH_BANK_OVERRIDE,
        research_find(lambda p, d, f: 'research_bank_manifest.json' in f), 'Research bank')
    helper = bank_root / 'source/_support/research_bank.py'
    spec = importlib.util.spec_from_file_location('research_bank_setup', helper)
    bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
    RESEARCH_ASSETS = bank.activate(bank_root)
    PROJECT_ROOT = RESEARCH_ASSETS['source_root']
    # Compatibility for existing data-audit publishing cells only.
    OFFLINE_ASSETS = {'root': bank_root, 'source_root': PROJECT_ROOT}
else:
    PROJECT_ROOT = research_one(globals().get('PROJECT_ROOT_OVERRIDE'),
        research_find(lambda p, d, f: (p / 'clindistill/research').is_dir() and 'configs' in d), 'Research source')
    sys.path.insert(0, str(PROJECT_ROOT))
print('Research source:', PROJECT_ROOT)


In [ ]:
from clindistill.artifacts import read_json
from clindistill.research.data import prepare_features
PREPARED_ROOT_OVERRIDE = None
IMAGE_ROOT_OVERRIDE = None
RESEARCH_FEATURE_OUTPUT_OVERRIDE = None
MODEL_LOCATIONS_OVERRIDE = None
IMAGE_BATCH_SIZE = 'auto'  # >=70 GiB free: B=256, L=128, ResNet=256, Swin=128
IMAGE_WORKERS = 'auto'  # up to 8 CPU workers; CPU software checks use 0
CLINICAL_BATCH_SIZE = 'auto'  # 128 on a largely free 96-GB GPU; FP32 text encoder
TEST_PROTOCOL_PATH = None
PREPARE_SPLITS = ['train', 'validation']
prepared = research_one(PREPARED_ROOT_OVERRIDE,
    [release_root] if 'release_root' in globals() else research_find(lambda p,d,f:
        'release.json' in f and not read_json(p / 'release.json').get('debug_subset')), 'Prepared release')
images = research_one(IMAGE_ROOT_OVERRIDE, research_find(lambda p,d,f:
    ((p / 'Images/Photographs').is_dir() or ('images' in d and 'cls_train.json' in f))
    and 'mini_dataset.json' not in f), 'Original CODe or Qwen data/images')
models = MODEL_LOCATIONS_OVERRIDE or (RESEARCH_ASSETS['models'] if RESEARCH_ASSETS else None)
if models is None:
    raise RuntimeError('Attach the ClinDistill research bank; Qwen weights are not DINO/BGE assets.')
cfg = read_json(PROJECT_ROOT / 'configs/code/research.json')
cfg.update(image_batch_size=IMAGE_BATCH_SIZE, image_workers=IMAGE_WORKERS, clinical_batch_size=CLINICAL_BATCH_SIZE)
destination = Path(RESEARCH_FEATURE_OUTPUT_OVERRIDE) if RESEARCH_FEATURE_OUTPUT_OVERRIDE else (
    Path('/kaggle/working/data/COde_processed/research_features_v3') if Path('/kaggle').exists()
    else PROJECT_ROOT.parent / 'Data/COde_processed/research_features_v3')
test_protocol = read_json(TEST_PROTOCOL_PATH) if TEST_PROTOCOL_PATH else None
manifest = prepare_features(prepared, images, models, destination, cfg, splits=PREPARE_SPLITS, test_protocol=test_protocol)
print('Research cache:', destination, '\nDimensions:', manifest['dimensions'], '\nSplits:', manifest['splits'])
print('Save Version and attach this Output to notebook02. Runtime: feature_runtime.json')
